In [1]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

In [3]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)

Train shape: (7613, 5)
Test shape: (3263, 4)
Sample submission shape: (3263, 2)


In [4]:
print("\nTrain columns:")
print(train_df.columns)

print("\nFirst 5 rows:")
print(train_df.head())


Train columns:
Index(['id', 'keyword', 'location', 'text', 'target'], dtype='object')

First 5 rows:
   id keyword location                                               text  \
0   1     NaN      NaN  Our Deeds are the Reason of this #earthquake M...   
1   4     NaN      NaN             Forest fire near La Ronge Sask. Canada   
2   5     NaN      NaN  All residents asked to 'shelter in place' are ...   
3   6     NaN      NaN  13,000 people receive #wildfires evacuation or...   
4   7     NaN      NaN  Just got sent this photo from Ruby #Alaska as ...   

   target  
0       1  
1       1  
2       1  
3       1  
4       1  


In [5]:
print("\nMissing values in train:")
print(train_df.isnull().sum())

print("\nMissing values in test:")
print(test_df.isnull().sum())


Missing values in train:
id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

Missing values in test:
id             0
keyword       26
location    1105
text           0
dtype: int64


In [6]:
print("\nClass balance:")
print(train_df["target"].value_counts())

print("\nClass balance percentage:")
print(train_df["target"].value_counts(normalize=True))


Class balance:
target
0    4342
1    3271
Name: count, dtype: int64

Class balance percentage:
target
0    0.57034
1    0.42966
Name: proportion, dtype: float64


In [7]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [8]:
train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

print("\nOriginal text example:")
print(train_df["text"].iloc[0])

print("\nCleaned text example:")
print(train_df["clean_text"].iloc[0])


Original text example:
Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all

Cleaned text example:
our deeds are the reason of this earthquake may allah forgive us all


In [9]:
X = train_df["clean_text"]
y = train_df["target"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\nSplit sizes:")
print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))


Split sizes:
Train: 5329
Validation: 1142
Test: 1142


In [10]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print("\nTF-IDF shapes:")
print("Train:", X_train_tfidf.shape)
print("Validation:", X_val_tfidf.shape)
print("Test:", X_test_tfidf.shape)


TF-IDF shapes:
Train: (5329, 5000)
Validation: (1142, 5000)
Test: (1142, 5000)


In [11]:
baseline_model = LogisticRegression(max_iter=1000)

baseline_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [12]:
val_pred = baseline_model.predict(X_val_tfidf)

print("\nValidation Results:")
print("Accuracy:", round(accuracy_score(y_val, val_pred), 4))
print("Precision:", round(precision_score(y_val, val_pred), 4))
print("Recall:", round(recall_score(y_val, val_pred), 4))
print("F1-score:", round(f1_score(y_val, val_pred), 4))

print("\nValidation Classification Report:")
print(classification_report(y_val, val_pred))

print("\nValidation Confusion Matrix:")
print(confusion_matrix(y_val, val_pred))


Validation Results:
Accuracy: 0.7995
Precision: 0.8149
Recall: 0.6904
F1-score: 0.7475

Validation Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.88      0.83       651
           1       0.81      0.69      0.75       491

    accuracy                           0.80      1142
   macro avg       0.80      0.79      0.79      1142
weighted avg       0.80      0.80      0.80      1142


Validation Confusion Matrix:
[[574  77]
 [152 339]]


In [13]:
test_pred = baseline_model.predict(X_test_tfidf)

print("\nTest Results:")
print("Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Precision:", round(precision_score(y_test, test_pred), 4))
print("Recall:", round(recall_score(y_test, test_pred), 4))
print("F1-score:", round(f1_score(y_test, test_pred), 4))

print("\nTest Classification Report:")
print(classification_report(y_test, test_pred))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))


Test Results:
Accuracy: 0.817
Precision: 0.8386
Recall: 0.7102
F1-score: 0.7691

Test Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.90      0.85       652
           1       0.84      0.71      0.77       490

    accuracy                           0.82      1142
   macro avg       0.82      0.80      0.81      1142
weighted avg       0.82      0.82      0.81      1142


Test Confusion Matrix:
[[585  67]
 [142 348]]


In [16]:
test_tfidf = vectorizer.transform(test_df["clean_text"])

final_predictions = baseline_model.predict(test_tfidf)

submission = pd.DataFrame({
    "id": test_df["id"],
    "target": final_predictions
})

submission.to_csv("baseline_submission.csv", index=False)

print("\nSubmission saved as:baseline_submission.csv")

print(submission.head())


Submission saved as:baseline_submission.csv
   id  target
0   0       1
1   2       0
2   3       1
3   9       1
4  11       1
